In [5]:
import os
import re
import time
from pathlib import Path
import pandas as pd
from tqdm import tqdm
from dotenv import load_dotenv


load_dotenv()

True

In [6]:
os.environ["HF_HOME"] = "/data/oussama/hf_cache"
os.environ["HUGGINGFACE_HUB_CACHE"] = "/data/oussama/hf_cache/hub"
os.environ["TRANSFORMERS_CACHE"] = "/data/oussama/hf_cache/transformers"

# Gemma 4 12B + LoRA adapter inference

This version keeps the same evaluation logic as your baseline Gemma notebook, but loads:

1. the original base model: `google/gemma-4-12B-it`
2. your LoRA adapter: e.g. `models/lora_r4_alpha8_lr2e-4`

Use the final adapter folder for inference, not the Trainer checkpoint folder, unless you specifically want to evaluate that checkpoint.


In [7]:
import gc
import torch
from transformers import AutoProcessor, BitsAndBytesConfig

try:
    from transformers import AutoModelForImageTextToText as GemmaModelClass
except Exception:
    try:
        from transformers import AutoModelForMultimodalLM as GemmaModelClass
    except Exception:
        from transformers import AutoModelForCausalLM as GemmaModelClass

from peft import PeftModel


def run_gemma4_lora_local_on_questions(
    input_csv,
    output_csv,
    prompt_col="Prompt",
    base_model_id="google/gemma-4-12B-it",
    adapter_path="lora_results/models/lora_r4_alpha8_lr2e-4",
    question_id_col=None,
    gold_col=None,
    level_col="level",
    medical_specialty_col="medical_specialty",
    max_retries=5,
    save_every=1,
    max_new_tokens=10,
    cache_dir="/data/oussama/hf_cache",
    offload_folder="/data/oussama/offload_gemma_lora_inference",
    use_4bit=True,
    gpu_memory="13GiB",
    cpu_memory="350GiB",
    local_files_only=False,
):
    """
    Run Gemma 4 12B locally with a saved LoRA adapter and save predictions.

    This uses the same output format as the baseline function:
    - question_id
    - gold
    - prediction
    - raw_output
    - correct
    - level
    - medical_specialty
    - model
    - stop_reason
    - refusal_category
    - refusal_type

    Important:
    adapter_path should point to the folder containing:
    - adapter_config.json
    - adapter_model.safetensors
    """

    input_csv = Path(input_csv)
    output_csv = Path(output_csv)
    output_csv.parent.mkdir(parents=True, exist_ok=True)
    Path(offload_folder).mkdir(parents=True, exist_ok=True)

    print(f"Loading base model: {base_model_id}")
    print(f"Loading LoRA adapter: {adapter_path}")

    # Load processor/tokenizer from the base model. The adapter only contains LoRA weights,
    # not the full Gemma processor.
    processor = AutoProcessor.from_pretrained(
        base_model_id,
        cache_dir=cache_dir,
        local_files_only=local_files_only,
        trust_remote_code=True,
    )

    if getattr(processor, "tokenizer", None) is not None:
        if processor.tokenizer.pad_token_id is None:
            processor.tokenizer.pad_token = processor.tokenizer.eos_token

    max_memory = None
    if torch.cuda.is_available():
        max_memory = {i: gpu_memory for i in range(torch.cuda.device_count())}
        max_memory["cpu"] = cpu_memory
        print("max_memory:", max_memory)

    load_kwargs = dict(
        cache_dir=cache_dir,
        device_map="auto",
        low_cpu_mem_usage=True,
        trust_remote_code=True,
        local_files_only=local_files_only,
        offload_folder=offload_folder,
    )

    if max_memory is not None:
        load_kwargs["max_memory"] = max_memory

    if use_4bit:
        load_kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
            bnb_4bit_use_double_quant=True,
        )

    # Newer Transformers prefers dtype; older versions require torch_dtype.
    try:
        base_model = GemmaModelClass.from_pretrained(
            base_model_id,
            dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
            **load_kwargs,
        )
    except TypeError:
        base_model = GemmaModelClass.from_pretrained(
            base_model_id,
            torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
            **load_kwargs,
        )

    # Attach the LoRA adapter to the base model.
    gemma_model = PeftModel.from_pretrained(
        base_model,
        adapter_path,
        is_trainable=False,
        local_files_only=local_files_only,
    )

    gemma_model.eval()
    gemma_model.config.use_cache = True

    if hasattr(gemma_model, "hf_device_map"):
        print("Device map:", gemma_model.hf_device_map)

    df = pd.read_csv(input_csv, encoding="utf-8-sig")

    if prompt_col not in df.columns:
        raise ValueError(f"Missing prompt column: {prompt_col}")

    if gold_col is None:
        raise ValueError(
            "Could not find the gold answer column. "
            "Please pass gold_col='your_column_name'."
        )

    output_columns = [
        "question_id",
        "gold",
        "prediction",
        "raw_output",
        "correct",
        "level",
        "medical_specialty",
        "model",
        "stop_reason",
        "refusal_category",
        "refusal_type",
    ]

    def clean_value(value):
        if pd.isna(value):
            return ""
        return str(value).strip()

    def normalize_gold(value):
        text = clean_value(value).upper()
        match = re.search(r"\b([A-F])\b", text)
        return match.group(1) if match else text[:1]

    def extract_prediction(raw_output, allowed_letters=None):
        raw_output = clean_value(raw_output).upper()

        if allowed_letters:
            allowed_letters = [x.upper() for x in allowed_letters]
        else:
            allowed_letters = ["A", "B", "C", "D", "E", "F"]

        if raw_output in allowed_letters:
            return raw_output

        pattern = r"\b(" + "|".join(map(re.escape, allowed_letters)) + r")\b"
        match = re.search(pattern, raw_output)
        if match:
            return match.group(1)

        if raw_output and raw_output[0] in allowed_letters:
            return raw_output[0]

        return ""

    def parse_gemma_output(decoded_text):
        text = decoded_text

        try:
            parsed = processor.parse_response(decoded_text)

            if isinstance(parsed, str):
                text = parsed
            elif isinstance(parsed, dict):
                text = (
                    parsed.get("content")
                    or parsed.get("answer")
                    or parsed.get("text")
                    or str(parsed)
                )
            elif isinstance(parsed, list) and len(parsed) > 0:
                first = parsed[0]
                if isinstance(first, dict):
                    text = first.get("content") or first.get("text") or str(first)
                else:
                    text = str(first)
            else:
                text = str(parsed)

        except Exception:
            text = decoded_text

        text = clean_value(text)

        # Remove common special tokens if they appear.
        text = text.replace("<turn|>", " ")
        text = text.replace("<end_of_turn>", " ")
        text = text.replace("<start_of_turn>", " ")
        text = text.strip()

        return text

    def get_input_device(model):
        """
        With device_map='auto', put input_ids on the same GPU as the input embeddings.
        This avoids cuda:0/cuda:3 mismatches on multi-GPU sharded Gemma.
        """
        try:
            return model.get_input_embeddings().weight.device
        except Exception:
            pass

        try:
            return next(model.parameters()).device
        except Exception:
            pass

        return torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

    def move_inputs_to_device(inputs, device):
        if hasattr(inputs, "to"):
            return inputs.to(device)
        return {
            k: v.to(device) if torch.is_tensor(v) else v
            for k, v in inputs.items()
        }

    input_device = get_input_device(gemma_model)
    print("Input device:", input_device)

    def call_model(prompt, allowed_letters=None, max_retries=5):
        """
        Calls adapted Gemma 4 locally.

        Returns:
        {
            "raw_output": str,
            "stop_reason": str,
            "refusal_category": None,
            "refusal_type": None,
        }
        """

        if allowed_letters is None:
            allowed_letters = ["A", "B", "C", "D", "E", "F"]

        allowed_text = ", ".join(allowed_letters)

        system_prompt = (
            "You are evaluating multiple-choice questions from a medical benchmark. "
            "Select the single best answer option. "
            "Return only one uppercase letter. "
            "Do not explain."
        )

        user_prompt = (
            f"{prompt}\n\n"
            f"Allowed options: {allowed_text}\n"
            "Return exactly one uppercase letter from the allowed options. "
            "No explanation."
        )

        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ]

        last_error = None

        for attempt in range(1, max_retries + 1):
            try:
                try:
                    inputs = processor.apply_chat_template(
                        messages,
                        tokenize=True,
                        return_dict=True,
                        return_tensors="pt",
                        add_generation_prompt=True,
                        enable_thinking=False,
                    )
                except TypeError:
                    inputs = processor.apply_chat_template(
                        messages,
                        tokenize=True,
                        return_dict=True,
                        return_tensors="pt",
                        add_generation_prompt=True,
                    )

                inputs = move_inputs_to_device(inputs, input_device)
                input_len = inputs["input_ids"].shape[-1]

                with torch.inference_mode():
                    outputs = gemma_model.generate(
                        **inputs,
                        max_new_tokens=max_new_tokens,
                        do_sample=False,
                        pad_token_id=processor.tokenizer.eos_token_id,
                    )

                generated_tokens = outputs[0][input_len:]

                decoded = processor.decode(
                    generated_tokens,
                    skip_special_tokens=False,
                )

                raw_output = parse_gemma_output(decoded)

                if raw_output:
                    return {
                        "raw_output": raw_output,
                        "stop_reason": "local_lora_generate",
                        "refusal_category": None,
                        "refusal_type": None,
                    }

                print("Empty adapted Gemma response:")
                print("decoded:", repr(decoded))

                raise RuntimeError("Adapted Gemma returned empty text output.")

            except Exception as e:
                last_error = e

                if attempt == max_retries:
                    break

                wait_time = min(2 ** attempt, 60)
                print(
                    f"Adapted local Gemma error attempt {attempt}/{max_retries}: {repr(e)}. "
                    f"Retrying in {wait_time}s..."
                )
                time.sleep(wait_time)

        raise RuntimeError(
            f"Failed after {max_retries} retries. Last error: {repr(last_error)}"
        )

    # Read already processed question IDs if results CSV exists.
    processed_ids = set()

    if output_csv.exists():
        old_results = pd.read_csv(output_csv, encoding="utf-8-sig")

        if "question_id" in old_results.columns:
            processed_ids = set(old_results["question_id"].astype(str))
        elif "Question_id" in old_results.columns:
            processed_ids = set(old_results["Question_id"].astype(str))

        for col in output_columns:
            if col not in old_results.columns:
                old_results[col] = ""

        old_results = old_results[output_columns]
        results = old_results.to_dict("records")
    else:
        old_results = pd.DataFrame(columns=output_columns)
        results = []

    print(f"Already processed: {len(processed_ids)} questions")

    newly_processed = 0
    model_label = f"{base_model_id}+LoRA:{adapter_path}"

    for idx, row in tqdm(df.iterrows(), total=len(df)):
        question_id = (
            clean_value(row[question_id_col])
            if question_id_col is not None
            else str(idx)
        )

        if str(question_id) in processed_ids:
            continue

        gold = normalize_gold(row[gold_col])

        group = clean_value(row["Group"]).upper() if "Group" in df.columns else "ABCDEF"

        allowed_letters = [
            letter for letter in group
            if letter in ["A", "B", "C", "D", "E", "F"]
        ]

        if not allowed_letters:
            allowed_letters = ["A", "B", "C", "D", "E", "F"]

        prompt = clean_value(row[prompt_col])

        call_result = call_model(
            prompt=prompt,
            allowed_letters=allowed_letters,
            max_retries=max_retries,
        )

        raw_output = call_result["raw_output"]
        stop_reason = call_result["stop_reason"]
        refusal_category = call_result["refusal_category"]
        refusal_type = call_result["refusal_type"]

        prediction = extract_prediction(raw_output, allowed_letters)
        correct = int(prediction == gold)

        result = {
            "question_id": question_id,
            "gold": gold,
            "prediction": prediction,
            "raw_output": raw_output,
            "correct": correct,
            "level": clean_value(row[level_col]) if level_col in df.columns else "",
            "medical_specialty": (
                clean_value(row[medical_specialty_col])
                if medical_specialty_col in df.columns
                else ""
            ),
            "model": model_label,
            "stop_reason": stop_reason,
            "refusal_category": refusal_category,
            "refusal_type": refusal_type,
        }

        results.append(result)
        processed_ids.add(str(question_id))
        newly_processed += 1

        if save_every and newly_processed % save_every == 0:
            pd.DataFrame(results, columns=output_columns).to_csv(
                output_csv,
                index=False,
                encoding="utf-8-sig",
            )

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()

    results_df = pd.DataFrame(results, columns=output_columns)

    results_df.to_csv(
        output_csv,
        index=False,
        encoding="utf-8-sig",
    )

    return results_df


In [ ]:
# Run adapted Gemma 4 12B IT with your final LoRA adapter.
# Use the final adapter folder for inference:
ADAPTER_PATH = "lora_results/models/lora_r4_alpha8_lr2e-4"

# If you specifically want to evaluate the trainer checkpoint instead, use this:
# ADAPTER_PATH = "trainer_lora_r4_alpha8_lr2e-4/checkpoint-590"

results_lora = run_gemma4_lora_local_on_questions(
    input_csv="data/cleaned/medarabench_test_with_prompts.csv",
    output_csv="results/gemma-4-12B-it_lora_r4_alpha8_lr2e-4_results.csv",
    prompt_col="Prompt",
    base_model_id="google/gemma-4-12B-it",
    adapter_path=ADAPTER_PATH,
    question_id_col="Question Number",
    gold_col="Correct Answer",
    level_col="Level",
    medical_specialty_col="Medical Specialty",
    save_every=1,
    max_new_tokens=10,
    use_4bit=True,
    local_files_only=True,  # set True if the base model is already fully cached and you want offline mode
)

results_lora.head()


Loading base model: google/gemma-4-12B-it
Loading LoRA adapter: lora_results/models/lora_r4_alpha8_lr2e-4
max_memory: {0: '13GiB', 1: '13GiB', 2: '13GiB', 3: '13GiB', 'cpu': '350GiB'}


Loading weights:   0%|          | 0/677 [00:00<?, ?it/s]

Input device: cuda:3
Already processed: 0 questions


  9%|▊         | 427/4959 [13:08<2:19:33,  1.85s/it]


KeyboardInterrupt: 

: 

In [ ]:
# Quick summary: accuracy and invalid prediction rate.
VALID = set("ABCDEF")

summary = pd.DataFrame([{
    "model": results_lora["model"].iloc[0] if len(results_lora) else "Gemma 4 12B + LoRA",
    "n": len(results_lora),
    "accuracy": results_lora["correct"].mean(),
    "invalid_rate": (~results_lora["prediction"].astype(str).str.upper().isin(VALID)).mean(),
}])

summary
